In [247]:
import csv

def read_csv_to_biz_info(csv_file):
    biz_infos = []
    with open(csv_file, 'r') as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            name, phone, zip_code = row
            address = None
            wi_fi = None
            alcohol = None
            zip_code = zip_code
            meta = None
            biz_info = BizInfo(name, address, phone, wi_fi, alcohol, zip_code, meta)
            biz_infos.append(biz_info)
    return biz_infos

csv_file = 'business_list.csv'

the_list = read_csv_to_biz_info(csv_file)

In [248]:
print(len(the_list))

98


In [246]:
class BizInfo:
    def __init__(self, name, address=None, phone=None, wi_fi=None, alcohol=None, zip_code=None, meta = {}):
        self.name = name
        self.address = address
        self.phone = phone
        self.wi_fi = wi_fi
        self.alcohol = alcohol
        self.zip_code = zip_code
        self.meta = {}

# NOT ADDED
    def __repr__(self):
        return f"BizInfo(name={self.name}, address={self.address}, phone={self.phone}, wi_fi={self.wi_fi}, alcohol={self.alcohol})"

    def __str__(self):
        return f"{self.name}\nAddress: {self.address}\nPhone: {self.phone}\nWi-Fi: {self.wi_fi}\nAlcohol: {self.alcohol}\nZipCode: {self.zip_code}"

class DataPoint:
    def __init__(self, name, strategy, selector, is_bool = False):
        self.name = name
        self.strategy = strategy
        self.selector = selector
        self.is_bool = is_bool

In [216]:
import abc 

class BizScraper(abc.ABC):
    def __init__(self):
        self.datapoints = {}
        
    @abc.abstractmethod
    def load_page_for_biz(biz: str):
        pass
        
    @abc.abstractmethod
    def _retrieve_datapoint_value(datapoint_name: str, boolean=False) -> str:
        '''
        This is private as the precondition for this method is: load_page_for_biz
        '''
        pass

    def update_biz_info(self, biz: BizInfo) -> BizInfo:
        self.load_page_for_biz(biz.name)
        
        attribute_names = list(biz.__dict__.keys())
        attribute_names.remove('meta')
        
        for attr_name in attribute_names:
            if getattr(biz, attr_name) is None:
                setattr(biz, attr_name, self._retrieve_datapoint_value(attr_name))
                
        return biz


class FailedToFindBizContentError(Exception):
    pass


In [255]:

class YelpSeleniumScraper(BizScraper):
    _LANDING_PAGE = 'https://www.yelp.com/'
    _TIMEOUT = 1
    _TIMEOUT_LONG = 2
    _COOKIES_ID_BTN = 'onetrust-accept-btn-handler'
    BIZ_LOCATION = 'San Francisco Bay Area, CA'
    
    def __init__(self):
        # init the datapoints
        self.datapoints = {}
        self.datapoints['address'] = DataPoint(name='address', strategy=By.TAG_NAME, selector='address')
        self.datapoints['phone'] = DataPoint(name='phone', strategy=By.XPATH, selector="//div[p[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'phone number')]]/p[@data-font-weight='semibold']")
        self.datapoints['wi_fi'] = DataPoint(name='wi_fi', strategy=By.XPATH, selector="//span[text()='Free Wi-Fi']", is_bool = True)
        self.datapoints['alcohol'] = DataPoint(name='alcohol', strategy=By.XPATH, selector="//span[text()='Full Bar']", is_bool = True)

        # init selenium driver
        self.driver_init()
        

    def driver_init(self):
        options = webdriver.ChromeOptions()
        options.add_argument('--headless')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')

        self._driver = webdriver.Chrome(options=options)
        self._accept_cookies()
        
    def driver_quit(self):
        self._dirver.quit()

    def _accept_cookies(self):
        self._driver.get(self._LANDING_PAGE)
        time.sleep(self._TIMEOUT_LONG)

        try:
            button_locator = (By.ID, self._COOKIES_ID_BTN)
            accept_button = WebDriverWait(self._driver, self._TIMEOUT).until(EC.element_to_be_clickable(button_locator))
            if accept_button:
                accept_button.click()
        except TimeoutException:
            pass # already accepted
            
    def load_page_for_biz(self, biz_name: str):
        biz_link = self._retrieve_biz_link(biz_name)
        
        self._driver.get(link)

        # Expand aminities section for the biz page
        try:
            aminities_button = (By.XPATH, '//*[contains(text(), "More Attributes")]')
            expand_button = WebDriverWait(self._driver, self._TIMEOUT_LONG).until(
                EC.element_to_be_clickable(aminities_button)
            )
            expand_button.click()
            print(f'Expanded aminities for -> {biz_name}')
        except TimeoutException:
            print('No Amenities for this business')

    def _retrieve_datapoint_value(self, datapoint_name: str, boolean=False) -> str:
        dp = self.datapoints.get(datapoint_name)
        if dp is None:
            return None
            
        try:
            element = self._driver.find_element(dp.strategy, dp.selector)
            if dp.is_bool: return True

            return element.text
        except NoSuchElementException:
            if dp.is_bool: return False

            return None

    def _retrieve_biz_link(self, biz_name) -> Optional[str]: 
        self._navigate_to_landing()
        
        input_element = self._driver.find_element(By.ID, 'search_description')
        input_element.clear()
        input_element.send_keys(biz_name)
        
        input_element = self._driver.find_element(By.ID, 'search_location')
        input_element.clear()
        input_element.send_keys(self.BIZ_LOCATION)
        
        self._driver.find_element(By.XPATH, "//button[@aria-label='Search' and @type='submit']").click()
        time.sleep(self._TIMEOUT_LONG)

        try:
            link = self._driver.find_element(By.LINK_TEXT, biz_name).get_attribute("href")
            return link
        except Exception as ex:
            print(f"There was an exception while seraching for biz in Yelp: {ex}")
            raise FailedToFindBizContentError

    def _navigate_to_landing(self):
        self._driver.get('https://www.yelp.com')
        time.sleep(self._TIMEOUT)

In [256]:
s = YelpSeleniumBizScraper()

In [257]:
biz1 = BizInfo(name='Fournée Bakery')
biz2 = BizInfo(name='Radhaus')
biz3 = BizInfo(name='Skin by Maisha')

In [258]:
s.update_biz_info(biz3)

There was an exception while seraching for biz in Yelp: Message: no such element: Unable to locate element: {"method":"link text","selector":"Skin by Maisha"}
  (Session info: chrome-headless-shell=126.0.6478.127); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
0   chromedriver                        0x0000000101db7078 chromedriver + 5169272
1   chromedriver                        0x0000000101daef4a chromedriver + 5136202
2   chromedriver                        0x000000010192b36c chromedriver + 402284
3   chromedriver                        0x0000000101978740 chromedriver + 718656
4   chromedriver                        0x0000000101978a01 chromedriver + 719361
5   chromedriver                        0x00000001019bdbc4 chromedriver + 1002436
6   chromedriver                        0x000000010199badd chromedriver + 862941
7   chromedriver                        0x00000001019baf57

FailedToFindBizContentError: 

In [74]:

















import time
from typing import Optional
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import TimeoutException


In [231]:

# Set up the WebDriver
options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(options=options)

In [234]:
driver.get('https://www.yelp.com')
time.sleep(1)

try:
    button_locator = (By.ID, 'onetrust-accept-btn-handler')
    accept_button = WebDriverWait(driver, 1).until(EC.element_to_be_clickable(button_locator))
    if accept_button:
        accept_button.click()
except TimeoutException:
    # not found, do nothing
    pass

In [235]:
input_element = driver.find_element(By.ID, 'search_description')
input_element.clear()
input_text = "Radhaus"
input_element.send_keys(input_text)

input_element = driver.find_element(By.ID, 'search_location')
input_element.clear()
input_text = "San Francisco Bay Area, CA"
input_element.send_keys(input_text)

driver.find_element(By.XPATH, "//button[@aria-label='Search' and @type='submit']").click()
time.sleep(2)

# driver.find_element(By.LINK_TEXT, "Fournée Bakery").click()
link = driver.find_element(By.LINK_TEXT, "Radhaus").get_attribute("href")
driver.get(link)

address_element = driver.find_element(By.TAG_NAME, 'address')
address = address_element.text
print(f"Address: {address}")

Address: 2 Marina Blvd
Bldg A
San Francisco, CA 94123


In [236]:
# Enter function
# driver.get('https://www.yelp.com/biz/fourn%C3%A9e-bakery-berkeley?osq=Fourn%C3%A9e+Bakery')
# driver.get('https://www.yelp.com/biz/ma-la-london')

#driver.get('https://www.yelp.com/biz/stone-meadow-benefits-and-insurance-associates-redwood-city?osq=Stone+Meadow+Benefits+%26+Insurance+Associates&override_cta=Request+a+Quote')

# Accept cookies (only once)
# button_locator = (By.ID, 'onetrust-accept-btn-handler')
# button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable(button_locator))
# button.click()


# Expand amenities section
try:
    dynamic_locator = (By.XPATH, '//*[contains(text(), "More Attributes")]')
    expand_button = WebDriverWait(driver, 3).until(
        EC.element_to_be_clickable(dynamic_locator)
    )
    expand_button.click()
except TimeoutException:
    print('No Amenities for this business')

In [237]:
# name
# h1_element = WebDriverWait(driver, 1).until(EC.presence_of_element_located((By.TAG_NAME, 'h1')))
h1_element = driver.find_element(By.TAG_NAME, 'h1')
print(f"Name: {h1_element.text}")

# address
address_element = driver.find_element(By.TAG_NAME, 'address')
address = address_element.text
print(f"Address: {address}")

# phone
phone_number = driver.find_element(By.XPATH, "//div[p[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'phone number')]]/p[@data-font-weight='semibold']").text
print(f"Phone: {phone_number}")

# wi-fi
try:
    free_wifi_element = driver.find_element(By.XPATH, "//span[text()='Free Wi-Fi']")
    has_free_wifi = True
except NoSuchElementException:
    has_free_wifi = False
print(f"Wi-Fi: {has_free_wifi}")

# alcohol
try:
    free_wifi_element = driver.find_element(By.XPATH, "//span[text()='Full Bar']")
    has_alcohol = True
except NoSuchElementException:
    has_alcohol = False
print(f"Alcohol: {has_alcohol}")

Name: Radhaus
Address: 2 Marina Blvd
Bldg A
San Francisco, CA 94123
Phone: (415) 445-4556
Wi-Fi: False
Alcohol: True


In [100]:
# Finally
driver.quit()
